In [1]:
import random, math
import numpy as np
from tslearn.metrics import dtw, dtw_path
from timeit import default_timer as timer

In [2]:
# Setting up the inputs
# I do not know what happens with arrays more than 1D
# research DTW with 2D arrays

start = timer()

inputSize = 20
Alist = []
for i in range(inputSize):
#   randomx.append(random.randint(1,1000))
    Alist.append(i)
A = np.asarray(Alist)
print(A)

Blist = []
for i in range(inputSize, inputSize*2):
    #randomy.append(random.randint(1,1000))
    Blist.append(i)
B = np.asarray(Blist)
print(B)

print("DONE GENERATING")

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39]
DONE GENERATING


In [3]:
#Base DTW 
#dtw_score = dtw(x, y)
# Or, if the path is also an important information:
#optimal_path, dtw_score = dtw_path(x, y)

In [4]:
# Modified Dynamic Time Warping Start

In [68]:
# Generate the subsequences given a size g
# Parameters: subsequenceSize: the size of the subsequences, sequenceInput: the sequence to be split into subsets
def generateSubsequences(subsequenceSize, sequenceInput):
    numberOfSubsequences = math.floor(len(sequenceInput)/(subsequenceSize-1))
    numpyInput = np.asarray(sequenceInput)
    
    allSubsequences = np.zeros((numberOfSubsequences, subsequenceSize))
    
    for i in range(numberOfSubsequences):
        sequenceIndex = i*(subsequenceSize-1)
        newSubsequence = np.ones(subsequenceSize) * np.inf
        
        if i > 0:
            newSubsequence[0] = allSubsequences[i-1, -1]
        for j in range(subsequenceSize-1):
            newSubsequence[j+1] = sequenceInput[sequenceIndex]
            sequenceIndex += 1
        allSubsequences[i] = newSubsequence

    return allSubsequences

# Subsequence size of 5 for sequences A and B
subseqSizeG = 5
allSubsequencesA = generateSubsequences(subseqSizeG, A)
allSubsequencesB = generateSubsequences(subseqSizeG, B)
print(allSubsequencesA)
print(allSubsequencesB)

[[inf  0.  1.  2.  3.]
 [ 3.  4.  5.  6.  7.]
 [ 7.  8.  9. 10. 11.]
 [11. 12. 13. 14. 15.]
 [15. 16. 17. 18. 19.]]
[[inf 20. 21. 22. 23.]
 [23. 24. 25. 26. 27.]
 [27. 28. 29. 30. 31.]
 [31. 32. 33. 34. 35.]
 [35. 36. 37. 38. 39.]]


In [99]:
b, a = np.meshgrid(allSubsequencesB[2], allSubsequencesA[1])
distanceMatrix = np.abs(a - b)
print(distanceMatrix)
print(a)
print(b)

[[24. 25. 26. 27. 28.]
 [23. 24. 25. 26. 27.]
 [22. 23. 24. 25. 26.]
 [21. 22. 23. 24. 25.]
 [20. 21. 22. 23. 24.]]
[[3. 3. 3. 3. 3.]
 [4. 4. 4. 4. 4.]
 [5. 5. 5. 5. 5.]
 [6. 6. 6. 6. 6.]
 [7. 7. 7. 7. 7.]]
[[27. 28. 29. 30. 31.]
 [27. 28. 29. 30. 31.]
 [27. 28. 29. 30. 31.]
 [27. 28. 29. 30. 31.]
 [27. 28. 29. 30. 31.]]


In [69]:
# Compute the cost matrix given two sequences (or subsequences)

# Maybe have some sort of firstSubsequence variable to signify if it is first subsequence from A or B
def computeDistanceMatrix(A, B, subsequenceSize):
    distanceMatrix = np.zeros((subsequenceSize, subsequenceSize))

    # Calculate distance matrix. 
    b, a = np.meshgrid(B,A)
    distanceMatrix = np.abs(a - b)
    return distanceMatrix

In [96]:
# Calculate all distance matrices with the the indices of the subsequences as a tuple key.
# The warning produced "RuntimeWarning: invalid value encountered in subtract distanceMatrix = np.abs(a - b)" is intentional
    # the warning is because of np.inf - np.inf in the first box

def calculateAllDistanceMatrices(subsequencesA, subsequencesB, subsequenceSize):
    subseqCountA = len(subsequencesA)
    subseqCountB = len(subsequencesB)
    
    allDistanceMatrices = {}
    for i in range(subseqCountA):
        for j in range(subseqCountB):
            key = (i, j)
            value = computeDistanceMatrix(subsequencesA[i], subsequencesB[j], subsequenceSize)
            if key == (0,0):
                value[(0,0)] = 0
            allDistanceMatrices[key] = value

    return allDistanceMatrices

distanceMatrices = calculateAllDistanceMatrices(allSubsequencesA, allSubsequencesB, subseqSizeG)
# print(len(distanceMatrices), distanceMatrices[(1,4)])
print(distanceMatrices[(0,0)])
print(distanceMatrices[(1,0)])
print(distanceMatrices[(1,1)])

[[ 0. inf inf inf inf]
 [inf 20. 21. 22. 23.]
 [inf 19. 20. 21. 22.]
 [inf 18. 19. 20. 21.]
 [inf 17. 18. 19. 20.]]
[[inf 17. 18. 19. 20.]
 [inf 16. 17. 18. 19.]
 [inf 15. 16. 17. 18.]
 [inf 14. 15. 16. 17.]
 [inf 13. 14. 15. 16.]]
[[20. 21. 22. 23. 24.]
 [19. 20. 21. 22. 23.]
 [18. 19. 20. 21. 22.]
 [17. 18. 19. 20. 21.]
 [16. 17. 18. 19. 20.]]


C:\Users\user\AppData\Local\Temp\ipykernel_10224\2460545317.py:9: RuntimeWarning: invalid value encountered in subtract
  distanceMatrix = np.abs(a - b)


In [71]:
# Keys for L and R
def calculateLRIndices(subsequenceSize):
    horizontalL = []
    verticalL = []
    
    horizontalR = []
    verticalR = []
    for i in range(subsequenceSize-1, -1, -1):
        # Indices for L
        indexHorL = (0, i)
        horizontalL.append(indexHorL)
        
        # Indices for R
        indexHorR = (subsequenceSize-1, i)
        horizontalR.append(indexHorR)
    
    for i in range(subsequenceSize):
        # Indices for L
        indexVerL = (i, 0)
        verticalL.append(indexVerL)
    
        # Indices for R
        indexVerR = (i, subsequenceSize-1)
        verticalR.append(indexVerR)
    
    # the pop removes the duplicate entry where the horizontal and vertical touch
    horizontalL.pop()
    indicesL = tuple(horizontalL + verticalL)
    
    verticalR.pop()
    indicesR = tuple(verticalR + horizontalR)
    
    print("L : ", indicesL)
    #for i in indicesL:
    #    print(distanceMatrices[(1,4)][i[0], i[1]])
    
    print("R : ", indicesR)
    #for i in keysR:
    #    print(distanceMatrices[(1,4)][i[0], i[1]])

    return indicesL, indicesR


pointsL, pointsR = calculateLRIndices(subseqSizeG)
print("L: \n", pointsL)
print("R: \n", pointsR)

L :  ((0, 4), (0, 3), (0, 2), (0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
R :  ((0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (4, 3), (4, 2), (4, 1), (4, 0))
L: 
 ((0, 4), (0, 3), (0, 2), (0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
R: 
 ((0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (4, 3), (4, 2), (4, 1), (4, 0))


In [72]:
# Enumerate all admissible pairs of positions

def enumeratePairsOfPositions(indicesL, indicesR):
    pairsOfPositions = []
    for i in indicesL:
        for j in indicesR:
            if i != j and i[0] <= j[0] and i[1] <= j[1]:
                vw = (i, j)
                pairsOfPositions.append(vw)
    
    return pairsOfPositions


admissiblePairs = enumeratePairsOfPositions(pointsL, pointsR)
print(len(admissiblePairs), admissiblePairs[0])

59 ((0, 4), (1, 4))


In [73]:
# Checks for all absolute possible moves for the node (ignoring pair of position restrictions)
def checkNextNodes(currentNode, subsequenceSize):
    nextNodes = []
    
    # Check for Horizontal Move
    if currentNode[1] < subsequenceSize-1:
        nextNodes.append((currentNode[0], currentNode[1]+1))
    # Check for Vertical Move
    if currentNode[0] < subsequenceSize-1:
        nextNodes.append((currentNode[0]+1, currentNode[1]))
    # Check for Diagonal Move, diagonal move is only possible if vertical and horizontal moves are possible
    if len(nextNodes) == 2:
        nextNodes.append((currentNode[0]+1, currentNode[1]+1))

    # Convert final list into a numpy array
    #numpyArray = np.asarray(nextNodes).astype(int)
    #return numpyArray
    return nextNodes

# Get all possible movements from the nodes into a dictionary
def enumerateAllNodeMoves(subsequenceSize):
    nodeMoves = {}
    for i in range(subsequenceSize):
        for j in range(subsequenceSize):
            currentNode = (i,j)
            nextNodes = checkNextNodes(currentNode, subsequenceSize)
            nodeMoves[currentNode] = nextNodes

    return nodeMoves

allNodeMoves = enumerateAllNodeMoves(subseqSizeG)

print(len(allNodeMoves))
#for key, value in allNodeMoves.items():
    #print(f"{key}: {len(value)}")

print(allNodeMoves[(0,0)])

25
[(0, 1), (1, 0), (1, 1)]


In [74]:
# Enumerate all possible paths for each pair of positions

def enumerateAllPossiblePaths(pairsOfPositions, nodeMoves):
    pathsDict = {}
    # reference: L: (0,3) and R: (3,4) (7 paths)
    for pair in pairsOfPositions:
        # print("CURRENT PAIR: ", pair)
        currentNode = pair[0]
        endNode = pair[1]
        
        allPaths = []
        unfinishedPaths = []

        currentPairPaths = {}
        dictKey = 0
    
        # Check starting node's moves if they are legal
        for move in nodeMoves[currentNode]:
            if move[0] <= endNode[0] and move[1] <= endNode[1]:
                unfinishedPaths.append([move])
            if move[0] == endNode[0] and move[1] == endNode[1]:
                allPaths.append(move)
                currentPairPaths[dictKey] = np.array(move)
                dictKey += 1
                unfinishedPaths.remove([move])
        #print(f"Unfinished paths: {unfinishedPaths}")
        #print(f"Completed paths: {allPaths}")
                
        while len(unfinishedPaths) > 0:
            # print(unfinishedPaths)
            currentPath = unfinishedPaths[0]
        
            raw_moves = nodeMoves[currentPath[-1]]
            moves = [x for x in raw_moves if x[0] <= endNode[0] and x[1] <= endNode[1]]
            # If there are more than 1 possible moves, make copies of the current path and append the moves
            while len(moves) > 1:
                pathCopy = currentPath.copy()
                pathCopy.append(moves[0])
                unfinishedPaths.append(pathCopy)
                moves.pop(0)
            currentPath.append(moves[0])
    
            # If the move is the end node already, put the path in allPaths to mark it completed
            if moves[0] == endNode:
                #print("POP IT")
                allPaths.append(currentPath)
                currentPairPaths[dictKey] = np.array(currentPath)
                dictKey += 1
                unfinishedPaths.pop(0)
                
            #print(f'Current: {currentPath}')
            #print(f"Unfinished: {unfinishedPaths}")
        pathsDict[pair] = currentPairPaths

        #print("_"*60, "RESULTS", "_"*60)
        # print(allPaths)
        #for key, value in currentPairPaths.items():
            #print(f"{key}: {len(value)}")
            #print(f"{key}: {value}")
        #print("_"*60, "END", "_"*60)
    return pathsDict
        
#for key, value in pathsDict.items():
    #print(f"{key}: {len(value)}")
    #print(f"{key}: {value}")
allPossiblePaths = enumerateAllPossiblePaths(admissiblePairs, allNodeMoves)
print("((0,3), (2,4))")
print(allPossiblePaths[((0,3), (2,4))])
print(allPossiblePaths[((0,3), (2,4))][0])
print(allPossiblePaths[((0,3), (2,4))][0][:, 0])
#for key, value in allPossiblePaths.items():
    # print(f"{key}: {len(value)}")
    # print(f"{key}: {value}")

((0,3), (2,4))
{0: array([[0, 4],
       [1, 4],
       [2, 4]]), 1: array([[1, 3],
       [2, 4]]), 2: array([[1, 4],
       [2, 4]]), 3: array([[1, 3],
       [1, 4],
       [2, 4]]), 4: array([[1, 3],
       [2, 3],
       [2, 4]])}
[[0 4]
 [1 4]
 [2 4]]
[0 1 2]


In [75]:
# STAGE 2 : Compact Dynamic Programming

In [116]:
print("L: \n", pointsL)
print("R: \n", pointsR)
print(distanceMatrices[(0,0)])
print(distanceMatrices[(0,1)])
print(distanceMatrices[(2,2)])
print(distanceMatrices[(2,2)][:, -1])
print(distanceMatrices[(2,2)][-1])

L: 
 ((0, 4), (0, 3), (0, 2), (0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
R: 
 ((0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (4, 3), (4, 2), (4, 1), (4, 0))
[[ 0. inf inf inf inf]
 [inf 20. 21. 22. 23.]
 [inf 19. 20. 21. 22.]
 [inf 18. 19. 20. 21.]
 [inf 17. 18. 19. 20.]]
[[inf inf inf inf inf]
 [23. 24. 25. 26. 27.]
 [22. 23. 24. 25. 26.]
 [21. 22. 23. 24. 25.]
 [20. 21. 22. 23. 24.]]
[[20. 21. 22. 23. 24.]
 [19. 20. 21. 22. 23.]
 [18. 19. 20. 21. 22.]
 [17. 18. 19. 20. 21.]
 [16. 17. 18. 19. 20.]]
[24. 23. 22. 21. 20.]
[16. 17. 18. 19. 20.]


In [170]:
# Support function for calculateCostMatrix() function
# calculates the minimal pairs using divide and conquer
def divideAndConquerMinimalPairs(costMatrix, minimalPairsDict, indicesL, indicesR, pathCosts):
    if len(indicesR) == 1:
        # print("BASE CASE HIT", indicesR[0], " | ", indicesL)
    
        # Find the minimal value of L
        lowestCostIndex = 0
        for i in range(len(indicesL)):
            index = indicesL[i]
            if costMatrix[indicesL[lowestCostIndex]]+pathCosts[indicesL[lowestCostIndex]] > costMatrix[index]+pathCosts[index]:
                lowestCostIndex = i

        # Set the index to be the lowest cost if it is not inf
        if costMatrix[indicesR[0][0]][indicesR[0][1]] != np.inf:
            lowestCost = costMatrix[indicesL[lowestCostIndex]]+pathCosts[indicesL[lowestCostIndex]]
            costMatrix[indicesR[0]] = lowestCost
            minimalPairsDict[indicesR[0]] = indicesL[lowestCostIndex]
            
            #print("BASE SET", indicesR[0], "TO", lowestCost)
        return costMatrix
    else:
        # Find median index k from R
        medianIndex = len(indicesR)//2
        midR = indicesR[medianIndex]

        # Find the minimal pair from L
        lowestCostIndex = 0

        # Find the minimal value of L
        for i in range(len(indicesL)):
            index = indicesL[i]
            if costMatrix[indicesL[lowestCostIndex]]+pathCosts[indicesL[lowestCostIndex]] > costMatrix[index]+pathCosts[index]:
                lowestCostIndex = i
                
        # set the current index to be the lowest cost if it is not inf
        if costMatrix[midR[0]][midR[1]] != np.inf:
            lowestCost = costMatrix[indicesL[lowestCostIndex]]+pathCosts[indicesL[lowestCostIndex]]
            costMatrix[midR] = lowestCost
            minimalPairsDict[midR] = indicesL[lowestCostIndex]
            # print("ELSE SET", midR, "TO", lowestCost)
        
        # Divide L and R into two new function calls
        # First Half
        firstHalfR = indicesR[0:medianIndex]
        firstHalfL = indicesL[0:lowestCostIndex+1]

        #print("ELSE HIT", medianIndex, midR)
        #print("First Half: ", firstHalfR, " | ", firstHalfL)
        divideAndConquerMinimalPairs(costMatrix, minimalPairsDict, firstHalfL, firstHalfR, pathCosts)

        # Second Half only if theres enough elements
        if len(indicesR) > 2:
            latterHalfR = indicesR[medianIndex+1:]
            latterHalfL = indicesL[lowestCostIndex:]
            # print("Latter Half: ", latterHalfR, " | ", latterHalfL)
            divideAndConquerMinimalPairs(costMatrix, minimalPairsDict, latterHalfL, latterHalfR, pathCosts)
            

    return costMatrix

In [171]:
def calculateAllCostMatrices(subsequenceSize,
                             subseqCountA,
                             subseqCountB,
                             indicesL, 
                             indicesR, 
                             pathCosts#,
                             #distanceMatrices, 
                             #distanceMatrixIndex
                            ):
    allCostMatrices = {}
    allMinimalPairs = {}

    # Start iterating per matrix 
    for i in range(subseqCountA):
        for j in range(subseqCountB):
            # set up values of L
            # If on an edge, set to inf, else copy from previous matrix
            currentMatrix = np.zeros((subsequenceSize, subsequenceSize))

            if i == 0:
                currentMatrix[0] = np.ones(subsequenceSize) * np.inf
            else:
                currentMatrix[0] = allCostMatrices[(i-1, j)][-1]
                
            if j == 0:
                currentMatrix[:, 0] = np.inf
            else:
                currentMatrix[:, 0] = allCostMatrices[(i, j-1)][:, -1]
                
            if (i == 0 and j == 0):
                currentMatrix[0][0] = 0
            
            # calculate values of R
            # 1. Calculate median index of R, k
            # 2. Find minimal pair of k where k is w in (u, w)
            # 3. Divide and Conquer the remaining minimal pairs
    
            currentMinPairs = {}
            # print("START", i, j)
            # print(currentMatrix)
            newCostMatrix = divideAndConquerMinimalPairs(currentMatrix, currentMinPairs, indicesL, indicesR, pathCosts)
            key = (i, j)
            allCostMatrices[key] = newCostMatrix
            allMinimalPairs[key] = currentMinPairs
                                                                                         
            print("END", i, j)
            print(allCostMatrices[key])
            print(len(allMinimalPairs[key]), allMinimalPairs[key])
            
    return allCostMatrices, allMinimalPairs

costs = {
    (0, 4):5, (0, 3):10, (0, 2):12, (0, 1):7, (0, 0):9, (1, 0):4, (2, 0):10, (3, 0):11, (4, 0):7
}


calculateAllCostMatrices(subseqSizeG, len(allSubsequencesA), len(allSubsequencesB), pointsL, pointsR, costs)#, distanceMatrices, (1, 1))

END 0 0
[[ 0. inf inf inf inf]
 [inf  0.  0.  0.  9.]
 [inf  0.  0.  0.  9.]
 [inf  0.  0.  0.  9.]
 [inf  9.  9.  9.  9.]]
7 {(4, 4): (0, 0), (2, 4): (0, 0), (1, 4): (0, 0), (3, 4): (0, 0), (4, 1): (0, 0), (4, 2): (0, 0), (4, 3): (0, 0)}
END 0 1
[[inf inf inf inf inf]
 [ 9.  0.  0.  0. 13.]
 [ 9.  0.  0.  0. 13.]
 [ 9.  0.  0.  0. 13.]
 [13. 13. 13. 13. 13.]]
8 {(4, 4): (1, 0), (2, 4): (1, 0), (1, 4): (1, 0), (3, 4): (1, 0), (4, 1): (1, 0), (4, 2): (1, 0), (4, 3): (1, 0), (4, 0): (1, 0)}
END 0 2
[[inf inf inf inf inf]
 [13.  0.  0.  0. 17.]
 [13.  0.  0.  0. 17.]
 [13.  0.  0.  0. 17.]
 [17. 17. 17. 17. 17.]]
8 {(4, 4): (1, 0), (2, 4): (1, 0), (1, 4): (1, 0), (3, 4): (1, 0), (4, 1): (1, 0), (4, 2): (1, 0), (4, 3): (1, 0), (4, 0): (1, 0)}
END 0 3
[[inf inf inf inf inf]
 [17.  0.  0.  0. 21.]
 [17.  0.  0.  0. 21.]
 [17.  0.  0.  0. 21.]
 [21. 21. 21. 21. 21.]]
8 {(4, 4): (1, 0), (2, 4): (1, 0), (1, 4): (1, 0), (3, 4): (1, 0), (4, 1): (1, 0), (4, 2): (1, 0), (4, 3): (1, 0), (4, 0): (1, 

({(0,
   0): array([[ 0., inf, inf, inf, inf],
         [inf,  0.,  0.,  0.,  9.],
         [inf,  0.,  0.,  0.,  9.],
         [inf,  0.,  0.,  0.,  9.],
         [inf,  9.,  9.,  9.,  9.]]),
  (0,
   1): array([[inf, inf, inf, inf, inf],
         [ 9.,  0.,  0.,  0., 13.],
         [ 9.,  0.,  0.,  0., 13.],
         [ 9.,  0.,  0.,  0., 13.],
         [13., 13., 13., 13., 13.]]),
  (0,
   2): array([[inf, inf, inf, inf, inf],
         [13.,  0.,  0.,  0., 17.],
         [13.,  0.,  0.,  0., 17.],
         [13.,  0.,  0.,  0., 17.],
         [17., 17., 17., 17., 17.]]),
  (0,
   3): array([[inf, inf, inf, inf, inf],
         [17.,  0.,  0.,  0., 21.],
         [17.,  0.,  0.,  0., 21.],
         [17.,  0.,  0.,  0., 21.],
         [21., 21., 21., 21., 21.]]),
  (0,
   4): array([[inf, inf, inf, inf, inf],
         [21.,  0.,  0.,  0., 25.],
         [21.,  0.,  0.,  0., 25.],
         [21.,  0.,  0.,  0., 25.],
         [25., 25., 25., 25., 25.]]),
  (1,
   0): array([[inf,  9.,  9.,

In [161]:
print(dtw_score)
print(optimal_path)

end = timer()
print(end-start)

NameError: name 'dtw_score' is not defined

In [ ]:
threeDArray = np.array([[[0,10], [1,20], [2,30]], [[4,50], [5,60], [6,70]], [[8,90], [9,100], [10,110]], [[12,130], [13,140], [14,150]]])

dummyarray = np.array([(1, 3), (2, 3), (3, 3), (3, 4)])
print(dummyarray)
print(dummyarray[:, 1])

print(threeDArray)

first_elements = threeDArray[:, :, 0]
print("First elements:")
print(first_elements)